# QAOA: подбор углов (h → γ, β) — финальное решение**Задача (Sber-чемпионат QAOA).** 12 кубитов, глубина p=5. Для каждого из 500 инстансов `h`(линейные коэффициенты Гамильтониана) подобрать 10 углов схемы QAOA — γ0..γ4 и β0..β4 —так, чтобы максимизировать **P(ground)**: вероятность, что схема (старт `|00..0⟩`)оказывается в основном (минимально-энергетическом) состоянии.Энергия: `E(s) = 0.5·sᵀJ·s + h·s`, s∈{±1}^12. Балл = среднее P по 500 инстансам.Ограничение финала: инференс на невидимых `h` — **≤ 10 минут**.**Результат: `merged2.csv` → mean P = 0.3380, 88/500 инстансов > 0.5** (лучший на лидерборде;для сравнения: случайные углы 0.006, обученный teacher конкурента 0.3175).

In [ ]:
import numpy as npimport torch# точный симулятор: QAOA (эталон) и FastQAOA (group=4, ~5x быстрее, расхождение < 1e-9)from QAOA import QAOA, FastQAOA, P   # P = глубина = 5

## 1. Данные* `J.npy` — матрица взаимодействий 12×12, **одна на все инстансы**.* `h_train.npy` — 500×12, инстансы лидерборда. `h_test.npy` — инференс-набор.* Углы — CSV `id,gamma_0..gamma_4,beta_0..beta_4` (500 строк).

In [ ]:
J = np.load("J.npy")            # (12,12)h = np.load("h_train.npy")      # (500,12)print("J:", J.shape, "h:", h.shape)

## 2. Симулятор (точный, без приближений)Схема p=5: 5 слоёв, в каждом — (а) фазовая часть `exp(-i·γ_l·H)` (H = Hamiltonian),(б) миксер `⊗_g exp(-i·β_l·X_g)` по группам. `FastQAOA` применяет миксер группами по 4 кубита(явная 16×16-матрица через popcount) — ровно тот же оператор, но в 5 раз быстрее.`p_ground` = сумма |амплитуд|² по состояниям с минимальной энергией (вырождение учитывается).

In [ ]:
q = FastQAOA(J, device="cpu" if not torch.cuda.is_available() else "cuda")ht = torch.tensor(h.astype(np.float32), device=q.device)# самопроверка: нулевые углы -> равномерное состояние -> P = 1/4096 (если основное не вырождено)zero = np.zeros((500, 10), dtype=np.float32)p0 = q.p_ground(ht, torch.tensor(zero[:, :P]), torch.tensor(zero[:, P:])).cpu().numpy()print("P(нулевые углы): min", p0.min(), "max", p0.max(), "(для невырожденного -> 1/4096 = 0.00024414)")

## 3. Методы (цепочка, всё — точный p_ground, никаких аппроксимаций)| этап | что делали | результат ||---|---|---|| family-сканы | структурированные семейства углов (uniform/wide/β-informed) | 0.05 → 0.324 || grind | локальный поиск по инстансам | 0.3354 || **lottery** | 4.1 млн случайных наборов (informed/uniform), **ratchet по каждому инстансу** | **0 побед** над merged2 → merged2 на локальных пиках || **merge/ratchet** | per-instance максимум из всех методов | **merged2 = 0.3380** || polish | координатный спуск по худшим 350 (сетки ±0.5→±0.015) | `polish_worst.py` || knn_warm | инференс h_test: K=8 соседей h_train + jitter + Adam | ≤ 10 мин |Ключевой факт: лотерея (2 млрд оценок) не нашла ни одной точки лучше merged2(`score_csv.py lotto_ratchet.csv merged2.csv` → 0/500). Дальше рост даёт толькодетерминированный локальный поиск (`polish_worst.py`) и перенос углов по похожим `h` (knn).

In [ ]:
# оценка любого CSV углов (встроенная копия score_csv.py):import csvdef load_angles(path):    d = np.genfromtxt(path, delimiter=",", names=True)    return np.stack([np.asarray(d[c], dtype=np.float32) for c in d.dtype.names if c != "id"], axis=1)A = load_angles("merged2.csv")pv = q.p_ground(ht, torch.tensor(A[:, :P]), torch.tensor(A[:, P:])).cpu().numpy()print(f"merged2: mean={pv.mean():.6f}  >0.5: {(pv>0.5).sum()}/500  max={pv.max():.5f}")top = np.argsort(-pv)[:5]print("top-5:", ", ".join(f"#{i}={pv[i]:.4f}" for i in top))

## 4. Инференс на h_test (финал, ≤ 10 минут)Для невидимых инстансов углы нельзя подбирать вручную — нужна функция h → углы.Используем эффект переноса: **похожие h → похожие углы**. Для каждого `h_test`:1. K=8 ближайших `h_train` по знаковой метрике `min(‖h−h′‖, ‖h+h′‖)` (учитывает симметрию P(h)=P(−h));2. берём углы соседей из лучшего CSV (`merged2.csv`), по 4 джиттер-копии (σ=0.08) — 40 кандидатов;3. дожимаем Adam по p_ground (~100 шагов), берём лучшее.```python knn_warm.py --h h_test.npy --targets merged2.csv --out submission_knn.csv# контроль (LOO-аналог на train, должно быть ~0.31+):python knn_warm.py --h h_train.npy --targets merged2.csv --out _selfcheck_train.csv```~5-9 минут на GPU — в 10-минутном бюджете с запасом.

## 5. Запуск и файлы```pip install -r requirements.txtpython score_csv.py  merged2.csv                 # оценка CSVpython scan_best.py                                # рейтинг всех CSV в папкеpython polish_worst.py --ref merged2.csv           # часовой улучшатель -> polish_ratchet.csvpython lottery.py --ref merged2.csv --sets 5000   # фоновая лотерея (ratchet-копилка)python make_viz_data.py                            # данные для 3D-визуализации```* `QAOA.py` — симуляторы (QAOA эталон / FastQAOA быстрый)* `score_csv.py`, `scan_best.py`, `polish_worst.py` — оценка/рейтинг/улучшение CSV* `lottery.py` — массовый поиск с ratchet; `knn_warm.py` — инференс h_test* `qaoa_viz.html` + `worker_sim.js` — 3D-визуализация (гора, 500 точек, 500 гор, полёт WASD)* `solution/` — pipeline модели «мозг» (features→model→train→distill→infer)* `presentation.pdf` — презентация; `colab_grind.ipynb` — кол-аб grind

## 6. Выводы1. **merged2 = 0.3380** — per-instance ретчет лучших углов из всех методов; 88/500 инстансов > 0.5.2. Лотерея 4.1 млн сидов: **0 побед** → merged2 уже на локальных пиках; случайный поиск исчерпан,   дальше — детерминированный спуск (`polish_worst.py`).3. Среднее тянут ~350 тяжёлых инстансов (P<0.3) — там и лежат оставшиеся очки.4. Для h_test — knn_warm (K=8 × jitter × Adam) в 10-минутном бюджете; оценка переноса по LOO: ~0.31.5. Верхняя граница среднего по открытым оценкам конкурентов: ~0.35-0.47.6. Всё посчитано **точным** симулятором (FastQAOA, group=4, сверено с эталоном до 1e-9).